# Activation functions: sigmoid, tanh and ReLU
Why a network needs a non-linear activation (a Keras experiment and the algebra), what "zero-centred"
changes in the gradients, and how fast each function is to compute.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"      # hide TensorFlow's start-up messages
import time
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
from sklearn.datasets import make_circles

tf.config.experimental.enable_op_determinism()   # same numbers on every run

## The three functions and their derivatives (saved for Figure 1)

In [2]:
z = np.linspace(-6, 6, 601)
sig = 1 / (1 + np.exp(-z))
tanh = np.tanh(z)
relu = np.maximum(0, z)
curves = pd.DataFrame({"z": z,
                       "sigmoid": sig, "d_sigmoid": sig * (1 - sig),      # sigmoid' = s(1 - s)
                       "tanh": tanh, "d_tanh": 1 - tanh ** 2,              # tanh' = 1 - tanh^2
                       "relu": relu, "d_relu": (z > 0).astype(float)})     # ReLU' = 0 or 1
curves.to_csv("data/activation_curves.csv", index=False)
print("largest derivative: sigmoid %.2f, tanh %.2f, ReLU %.0f"
      % (curves.d_sigmoid.max(), curves.d_tanh.max(), curves.d_relu.max()))
for v in (3, 5):
    print(f"at z = {v}: sigmoid' = {np.exp(-v) / (1 + np.exp(-v)) ** 2:.4f}, tanh' = {1 - np.tanh(v) ** 2:.4f}")

largest derivative: sigmoid 0.25, tanh 1.00, ReLU 1
at z = 3: sigmoid' = 0.0452, tanh' = 0.0099
at z = 5: sigmoid' = 0.0066, tanh' = 0.0002


## Non-linear data, linear activations
Two hidden layers of 32 nodes. First with `activation="linear"` (no activation at all), then with ReLU.

In [3]:
X, y = make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=2)
print(X.shape, np.bincount(y))

def build(activation, seed=0):
    keras.utils.set_random_seed(seed)
    model = keras.Sequential([
        keras.Input(shape=(2,)),
        keras.layers.Dense(32, activation=activation),
        keras.layers.Dense(32, activation=activation),
        keras.layers.Dense(1, activation="sigmoid")])     # sigmoid output: probability of class 1
    model.compile(optimizer=keras.optimizers.Adam(0.01), loss="binary_crossentropy", metrics=["accuracy"])
    return model

models = {}
for act in ("linear", "relu"):
    m = build(act)
    h = m.fit(X, y, epochs=200, batch_size=32, verbose=0)
    models[act] = m
    print(f"{act:6s}: loss {h.history['loss'][-1]:.3f}, training accuracy {h.history['accuracy'][-1]:.2f}")

(300, 2) [150 150]


linear: loss 0.694, training accuracy 0.50


relu  : loss 0.000, training accuracy 1.00


Probability of class 1 on a grid, for the decision-region figure.

In [4]:
g = np.linspace(-1.6, 1.6, 161)
gx, gy = np.meshgrid(g, g)
grid = np.c_[gx.ravel(), gy.ravel()]
out = pd.DataFrame({"x1": grid[:, 0], "x2": grid[:, 1]})
for act, m in models.items():
    out[act] = m.predict(grid, verbose=0).ravel()
out.to_csv("data/boundary_grid.csv", index=False, float_format="%.4g")
pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y}).to_csv("data/circles.csv", index=False)

## The linear network is one layer in disguise
Multiply out the three weight matrices: W' = W1 W2 W3 and b' = (b1 W2 + b2) W3 + b3 (Keras puts the
input on the left, x W). One layer with W' and b' gives exactly the same predictions.

In [5]:
(W1, b1), (W2, b2), (W3, b3) = [(l.kernel.numpy(), l.bias.numpy()) for l in models["linear"].layers]
W_ = W1 @ W2 @ W3
b_ = (b1 @ W2 + b2) @ W3 + b3
one_layer = 1 / (1 + np.exp(-(X @ W_ + b_)))
three_layers = models["linear"].predict(X, verbose=0)
print("W' =", W_.ravel().round(4), " b' =", b_.round(4))
print("largest difference between the two:", np.abs(one_layer - three_layers).max())

W' = [-0.0496 -0.0116]  b' = [-0.0611]
largest difference between the two: 5.686863724374547e-08


## Zero-centred or not: the sign of the gradients
One hidden layer of 10 nodes feeds one sigmoid output. For a single row, the gradient of each of the 10
weights into the output is (shared factor) x (activation of that hidden node).

In [6]:
def output_weight_grads(hidden_act, seed=1):
    keras.utils.set_random_seed(seed)
    m = keras.Sequential([keras.Input(shape=(2,)),
                          keras.layers.Dense(10, activation=hidden_act),
                          keras.layers.Dense(1, activation="sigmoid")])
    x, t = X[:1].astype("float32"), y[:1].astype("float32").reshape(-1, 1)
    with tf.GradientTape() as tape:
        loss = tf.reduce_mean(keras.losses.binary_crossentropy(t, m(x)))
    hidden = m.layers[0](x).numpy().ravel()
    grads = tape.gradient(loss, m.layers[1].kernel).numpy().ravel()
    return hidden, grads

for act in ("sigmoid", "tanh"):
    a, gr = output_weight_grads(act)
    print(f"{act}: activations {a.round(2)}")
    print(f"{' ' * len(act)}  gradient signs  {''.join('+' if v > 0 else '-' for v in gr)}")

sigmoid: activations [0.5  0.55 0.38 0.57 0.58 0.69 0.55 0.59 0.31 0.61]
         gradient signs  ++++++++++
tanh: activations [-0.    0.19 -0.44  0.29  0.3   0.66  0.2   0.33 -0.65  0.42]
      gradient signs  -+-+++++-+


Mean activation of a 128-node layer over all 300 rows (standardised inputs, Keras' default weights).

In [7]:
Xs = (X - X.mean(0)) / X.std(0)
for act in ("sigmoid", "tanh", "relu"):
    keras.utils.set_random_seed(0)
    layer = keras.layers.Dense(128, activation=act)
    a = layer(Xs.astype("float32")).numpy()
    print(f"{act:7s}: mean {a.mean():+.3f}, share of positive values {np.mean(a > 0):.2f}")

sigmoid: mean +0.500, share of positive values 1.00


tanh   : mean -0.000, share of positive values 0.50


relu   : mean +0.065, share of positive values 0.50


## Cost of computing each function
Time for 10 million values (numbers depend on the machine; the ratios matter).

In [8]:
big = np.random.default_rng(0).standard_normal(10_000_000)
for name, f in [("sigmoid", lambda v: 1 / (1 + np.exp(-v))), ("tanh", np.tanh), ("relu", lambda v: np.maximum(0, v))]:
    t0 = time.perf_counter()
    for _ in range(5):
        f(big)
    print(f"{name:7s}: {(time.perf_counter() - t0) / 5 * 1000:.0f} ms")

sigmoid: 3175 ms


tanh   : 1252 ms


relu   : 816 ms


## ReLU is not linear: two ReLUs make a bend (saved for Figure 4)

In [9]:
x = np.linspace(-3, 3, 601)
pd.DataFrame({"x": x, "a": np.maximum(0, x + 1), "b": np.maximum(0, x - 1),
              "diff": np.maximum(0, x + 1) - np.maximum(0, x - 1)}).to_csv("data/relu_bend.csv", index=False)

## ReLU's slope at exactly 0, and how often z is exactly 0
TensorFlow's gradient of ReLU at z = 0, then the share of exact zeros among all weighted sums
of the trained ReLU network's two hidden layers.

In [10]:
v = tf.constant([-1.0, 0.0, 1.0])
with tf.GradientTape() as tape:
    tape.watch(v)
    out = tf.nn.relu(v)
print("slope of ReLU at -1, 0, 1:", tape.gradient(out, v).numpy())
h, zs = X.astype("float32"), []
for layer in models["relu"].layers[:2]:
    zs.append((h @ layer.kernel.numpy() + layer.bias.numpy()).ravel())
    h = layer(h).numpy()
zs = np.concatenate(zs)
print(f"weighted sums: {zs.size}, exactly 0: {(zs == 0).sum()}")

slope of ReLU at -1, 0, 1: [0. 0. 1.]
weighted sums: 19200, exactly 0: 0


## Check: tanh(z) = 2 sigmoid(2z) - 1

In [11]:
print("largest difference:", np.abs(np.tanh(z) - (2 / (1 + np.exp(-2 * z)) - 1)).max())

largest difference: 3.3306690738754696e-16
